In [1]:
import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models
from tensorflow.keras.applications.inception_v3 import preprocess_input
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.applications import InceptionV3
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
import pandas as pd

In [3]:
TRAIN_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\train"
VALID_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\valid"
TEST_DIR = r"C:\Users\KristianHaltenJensen\Noroff\Bachelor\archive (5)\data – Kopi\test"

In [4]:
IMG_SIZE = (299, 299)
BATCH_SIZE = 32
EPOCHS_HEAD = 10
EPOCHS_FINE = 20
FINE_TUNE_AT = 249
LR_HEAD = 1e-3


In [5]:
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=20,
    width_shift_range=0.15,
    height_shift_range=0.15,
    zoom_range=0.15,
    shear_range=0.15,
    horizontal_flip=True,
    fill_mode="nearest"
)

valid_datagen = ImageDataGenerator(preprocessing_function=preprocess_input)
test_datagen  = ImageDataGenerator(preprocessing_function=preprocess_input)



In [6]:
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=True
)


valid_generator = valid_datagen.flow_from_directory(
    VALID_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    shuffle=False
)

Found 11167 images belonging to 12 classes.
Found 240 images belonging to 12 classes.
Found 600 images belonging to 12 classes.


In [7]:
num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())
print("Classes:", num_classes)
print(class_names)


Classes: 12
['Black Rust', 'Blast', 'Brown Rust', 'Common Root Rot', 'Fusarium Head Blight', 'Healthy', 'Leaf Blight', 'Mildew', 'Septoria', 'Smut', 'Tan spot', 'Yellow Rust']


In [8]:
base_model = InceptionV3(
    weights="imagenet",
    include_top=False,
    input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3)
)

base_model.trainable = False  # stage 1 

inputs = layers.Input(shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation="softmax")(x)

model = models.Model(inputs, outputs)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ input_layer_1 (InputLayer)           │ (None, 299, 299, 3)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ inception_v3 (Functional)            │ (None, 8, 8, 2048)          │      21,802,784 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d             │ (None, 2048)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 2048)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 12)                  │          24,588 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 21,827,372 (83.26 MB)

 Trainable params: 24,588 (96.05 KB)

 Non-trainable params: 21,802,784 (83.17 MB)

In [2]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    ),
    ModelCheckpoint(
        "ResNet50_transfer_best.keras",
        monitor="val_accuracy",
        save_best_only=True,
        verbose=1
    )
]

NameError: name 'EarlyStopping' is not defined

In [9]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_head = model.fit(
    train_generator,
    epochs=EPOCHS_HEAD,
    validation_data=valid_generator,
    callbacks = callbacks
)

C:\Users\KristianHaltenJensen\anaconda4\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 1094s 3s/step - accuracy: 0.1840 - loss: 12.2159 - val_accuracy: 0.2125 - val_loss: 6.8035
Epoch 2/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 876s 3s/step - accuracy: 0.2763 - loss: 6.2283 - val_accuracy: 0.2542 - val_loss: 5.4029
Epoch 3/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 925s 3s/step - accuracy: 0.3214 - loss: 4.7619 - val_accuracy: 0.2958 - val_loss: 4.7997
Epoch 4/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 764s 2s/step - accuracy: 0.3093 - loss: 4.4695 - val_accuracy: 0.2500 - val_loss: 6.3583
Epoch 5/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 6554s 19s/step - accuracy: 0.3210 - loss: 4.3292 - val_accuracy: 0.2542 - val_loss: 5.1536
Epoch 6/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 791s 2s/step - accuracy: 0.3316 - loss: 4.0243 - val_accuracy: 0.2250 - val_loss: 4.5208
Epoch 7/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 717s 2s/step - accuracy: 0.3145 - loss: 4.1237 - val_accuracy: 0.2250 - val_loss: 4.6545
Epoch 8/10
349/349 ━━━━━━━━━━━━━━━━━━━━ 602s 2s/step - accuracy: 0.3306 - loss: 4.0165 - val_

In [10]:
FINE_TUNE_AT = 249 

base_model.trainable = True
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

history_ft = model.fit(
    train_generator,
    epochs=EPOCHS_FINE,
    validation_data=valid_generator,
    verbose=1
)

Epoch 1/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 889s 2s/step - accuracy: 0.2237 - loss: 2.2550 - val_accuracy: 0.2167 - val_loss: 2.3657
Epoch 2/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 823s 2s/step - accuracy: 0.3810 - loss: 1.9130 - val_accuracy: 0.2833 - val_loss: 2.2907
Epoch 3/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 736s 2s/step - accuracy: 0.4494 - loss: 1.7224 - val_accuracy: 0.3375 - val_loss: 2.2247
Epoch 4/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 720s 2s/step - accuracy: 0.4926 - loss: 1.5992 - val_accuracy: 0.3542 - val_loss: 2.1486
Epoch 5/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 811s 2s/step - accuracy: 0.5073 - loss: 1.5231 - val_accuracy: 0.3625 - val_loss: 2.1024
Epoch 6/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 671s 2s/step - accuracy: 0.5322 - loss: 1.4562 - val_accuracy: 0.3875 - val_loss: 2.0134
Epoch 7/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 831s 2s/step - accuracy: 0.5591 - loss: 1.3634 - val_accuracy: 0.4208 - val_loss: 1.9428
Epoch 8/20
349/349 ━━━━━━━━━━━━━━━━━━━━ 837s 2s/step - accuracy: 0.5762 - loss: 1.3221 - val_accu

In [11]:
model.save('InceptionV3_WPD.keras')

In [12]:
import json
with open("training_history_InceptionV3.json", "w", encoding="utf-8") as f:
    json.dump(history_head.history, f, indent=2)



Saved: training_history_InceptionV3.json


In [14]:
test_loss, test_accuracy = model.evaluate(test_generator, verbose=1)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

19/19 ━━━━━━━━━━━━━━━━━━━━ 25s 1s/step - accuracy: 0.6241 - loss: 1.3918
Test loss: 1.4289
Test accuracy: 0.6467


In [16]:
import numpy as np

y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

y_true = test_generator.classes
class_names = list(test_generator.class_indices.keys())

19/19 ━━━━━━━━━━━━━━━━━━━━ 26s 1s/step 


In [17]:
from sklearn.metrics import classification_report

print(classification_report(
    y_true,
    y_pred,
    target_names=class_names
))


                           precision    recall  f1-score   support

          black_rust_test       0.86      0.60      0.71        50
               blast_test       0.78      0.86      0.82        50
          brown_rust_test       0.36      0.38      0.37        50
     common_root_rot_test       0.72      0.72      0.72        50
fusarium_head_blight_test       0.93      0.78      0.85        50
             healthy_test       0.10      0.04      0.06        50
         leaf_blight_test       0.67      0.60      0.63        50
              mildew_test       0.60      0.60      0.60        50
            septoria_test       0.73      0.90      0.80        50
                smut_test       0.87      0.96      0.91        50
            tan_spot_test       0.59      0.34      0.43        50
         yellow_rust_test       0.47      0.98      0.64        50

                 accuracy                           0.65       600
                macro avg       0.64      0.65      0.63    

In [21]:


cm = confusion_matrix(y_true, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=class_names,      
    columns=class_names     

print(cm_df)


                           black_rust_test  blast_test  brown_rust_test  \
black_rust_test                         30           1               12   
blast_test                               0          43                0   
brown_rust_test                          1           0               19   
common_root_rot_test                     0           3                2   
fusarium_head_blight_test                0           0                2   
healthy_test                             1           2                1   
leaf_blight_test                         1           0                6   
mildew_test                              0           1                7   
septoria_test                            0           0                0   
smut_test                                0           1                0   
tan_spot_test                            2           4                3   
yellow_rust_test                         0           0                1   

                        

In [22]:
cm_df

,black_rust_test,blast_test,brown_rust_test,common_root_rot_test,fusarium_head_blight_test,healthy_test,leaf_blight_test,mildew_test,septoria_test,smut_test,tan_spot_test,yellow_rust_test
black_rust_test,30,1,12,1,0,1,2,0,1,0,1,1
blast_test,0,43,0,1,0,2,1,0,0,2,1,0
brown_rust_test,1,0,19,2,0,3,1,11,6,0,0,7
common_root_rot_test,0,3,2,36,0,4,1,3,1,0,0,0
fusarium_head_blight_test,0,0,2,1,39,1,2,0,0,3,0,2
healthy_test,1,2,1,0,0,2,0,0,0,0,2,42
leaf_blight_test,1,0,6,1,1,2,30,1,1,1,5,1
mildew_test,0,1,7,1,0,2,0,30,4,0,3,2
septoria_test,0,0,0,1,0,1,3,0,45,0,0,0
smut_test,0,1,0,0,0,0,1,0,0,48,0,0


In [24]:
cm_df.to_csv("confusion_matrix_InceptionV3.csv")
print("Saved confusion_matrix_InceptionV3.csv")

Saved confusion_matrix_InceptionV3.csv
